In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
dataQ1_path = os.path.join(path, 'Q1_data.csv')
deliv_data = pd.read_csv(dataQ1_path)

print(f"Dataset shape: {deliv_data.shape}")

In [ ]:
# Task 2: Write your code here:
deliv_data.head()

In [ ]:
# Task 3: Write your code here:
deliv_data.info()
# Preparation_Time_min    Courier_Experience_yrs

In [ ]:
# Task 4: Write your code here:
deliv_data.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(deliv_data['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery time')
plt.xlabel('Minutes')
plt.ylabel('distance')
plt.show()

In [ ]:
# Task 1: Write your code here:
deliv_data = deliv_data.drop(columns=['Order_ID'])
deliv_data.head()

In [ ]:
# Task 2: Write your code here:
#missing = ['Weather'], ['Traffic_Level'], ['Time_of_Day'], ['Courier_Experience_yrs'], ['Delivery_Time']
cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time']

deliv_clean = deliv_data[cols].copy()
deliv_clean = deliv_clean.dropna(subset=['Delivery_Time','Courier_Experience_yrs'])
# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    deliv_clean[col] = deliv_clean[col].fillna('unknown')



In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(deliv_clean)

In [ ]:
# Task 4: Write your code here:
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day']
for col in categorical_cols:
    le = LabelEncoder()
    deliv_clean[col] = le.fit_transform(deliv_clean[col].astype(str))

deliv_clean.head()

In [ ]:
# Task 5: Write your code here:


In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:
feature_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs']
X = deliv_clean[feature_cols]
y = deliv_clean['Delivery_Time']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


#Training Random Forest:
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)

print("Model trained!")

# Kfold
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
mae_scores = np.array(mae_scores)
print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(y_fold_pred, bins=50, edgecolor='black')
plt.title('Delivery time')
plt.xlabel('Minutes')
plt.ylabel('distance')
plt.show()

In [ ]:
# Task Bonus: Write your code here: